# Tetris Fase 1A — Potencia en Colab

Este notebook clona el repo del proyecto, instala dependencias, corre las simulaciones para varios `tracker_prob`, y genera las curvas de potencia.

**Antes de empezar:** reemplaza `REPO_URL` en la celda siguiente por la URL de tu repo de GitHub.

## 1. Configuración

In [1]:
# CONFIGURACIÓN
REPO_URL = "https://github.com/yudocyudoc/tetris.git"
PROJECT_DIR = "/content/tetris"
N_GAMES = 50               # usar 100 si el tiempo y el runtime lo permiten
MAX_PIECES = 500
TAU = 10.0
H_MIN = 4
H_MAX = 15
TRACKER_PROBS = [0.10, 0.25, 0.50, 0.75]

import os
print("Configuración:")
print(f"  REPO_URL={REPO_URL}")
print(f"  N_GAMES={N_GAMES}")
print(f"  MAX_PIECES={MAX_PIECES}")

Configuración:
  REPO_URL=https://github.com/yudocyudoc/tetris.git
  N_GAMES=50
  MAX_PIECES=500


## 2. Clonar repo e instalar dependencias

In [2]:
import subprocess
import sys
import os

# Change to /content directory to ensure a stable working directory
os.chdir('/content')

# Remove repo if it exists
if os.path.isdir(PROJECT_DIR):
    !rm -rf {PROJECT_DIR}

# Clone repo without specifying destination, it will create 'tetris' in current dir
!git clone {REPO_URL}

# Change into the project directory
os.chdir(PROJECT_DIR)

# Install dependencies
!pip install -q pygame pandas pyarrow numpy scipy matplotlib tabulate statsmodels

Cloning into 'tetris'...
remote: Enumerating objects: 101, done.
remote: Counting objects: 100% (101/101), done.
remote: Compressing objects: 100% (54/54), done.
remote: Total 101 (delta 39), reused 99 (delta 37), pack-reused 0 (from 0)
Receiving objects: 100% (101/101), 135.13 KiB | 15.01 MiB/s, done.
Resolving deltas: 100% (39/39), done.


In [3]:
import os

print(f"Contents of {PROJECT_DIR}:")
!ls -F {PROJECT_DIR}

Contents of /content/tetris:
AGENTS.md				   main.py
analysis/				   MARCO_conceptual_agente_campo.md
BLUEPRINT_analisis_exploratorio.md	   README.md
BLUEPRINT_piso_confound_fase1A.md	   requirements.txt
BLUEPRINT_programa_tetris_agente_campo.md  run_exploratory_analysis.py
BLUEPRINT_tetris_instrumentado.md	   src/
colab/					   tests/
HANDOFF_programa_tetris.md


In [4]:
import os

confound_floor_path = os.path.join(PROJECT_DIR, "analysis", "confound_floor")
if os.path.isdir(confound_floor_path):
    print(f"El directorio '{confound_floor_path}' existe.")
else:
    print(f"El directorio '{confound_floor_path}' NO existe.")

El directorio '/content/tetris/analysis/confound_floor' existe.


In [5]:
print(f"Listado recursivo del contenido de {PROJECT_DIR}:")
!ls -R {PROJECT_DIR}

Listado recursivo del contenido de /content/tetris:
/content/tetris:
AGENTS.md				   main.py
analysis				   MARCO_conceptual_agente_campo.md
BLUEPRINT_analisis_exploratorio.md	   README.md
BLUEPRINT_piso_confound_fase1A.md	   requirements.txt
BLUEPRINT_programa_tetris_agente_campo.md  run_exploratory_analysis.py
BLUEPRINT_tetris_instrumentado.md	   src
colab					   tests
HANDOFF_programa_tetris.md

/content/tetris/analysis:
confound_floor

/content/tetris/analysis/confound_floor:
calibrate_signal_curve.py  fast_calibrate_signal_v2.py
confound_floor.py	   power_curve_t2.py
confound_floor_t2.py	   recalc_power_calibrated.py
fast_calibrate_signal.py

/content/tetris/colab:
README_colab.md  tetris_confound_colab.ipynb

/content/tetris/src:
exploratory_analysis.py  session_manager.py	tetris_ui.py
logger.py		 summarize_sessions.py	validate_session.py
metrics.py		 tetris_core.py

/content/tetris/tests:
test_headless_session.py  test_ui_input.py  test_ui_lifecycle.py


## 3. Verificar estructura

In [6]:
import os
conf_dir = os.path.join(PROJECT_DIR, "analysis", "confound_floor")
for f in ["confound_floor_t2.py", "power_curve_t2.py", "fast_calibrate_signal_v2.py", "recalc_power_calibrated.py"]:
    path = os.path.join(conf_dir, f)
    print("OK" if os.path.exists(path) else f"FALTA: {f}")

OK
OK
OK
OK


## 4. Correr simulaciones por tracker_prob

In [7]:
import os

ui_file_path = os.path.join(PROJECT_DIR, "src", "tetris_ui.py")
try:
    with open(ui_file_path, "r") as f:
        print("--- CONTENIDO DE tetris_ui.py ---")
        print(f.read())
except FileNotFoundError:
    print(f"Archivo no encontrado: {ui_file_path}")

--- CONTENIDO DE tetris_ui.py ---
"""Interfaz pygame del Tetris instrumentado.

Captura input crudo (keydown/keyup), implementa DAS/ARR, renderiza el juego
y orquesta las llamadas al motor y al logger.
"""

from __future__ import annotations

import time
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Tuple

import pygame

from .tetris_core import (
    BOARD_HEIGHT,
    BOARD_WIDTH,
    TetrisCoreGame,
    gravity_for_condition,
)
from .logger import SessionLogger

# ---------------------------------------------------------------------------
# Configuración de UI
# ---------------------------------------------------------------------------
CELL_SIZE = 30
BOARD_OFFSET_X = 50
BOARD_OFFSET_Y = 50
PREVIEW_OFFSET_X = BOARD_OFFSET_X + BOARD_WIDTH * CELL_SIZE + 40
PREVIEW_OFFSET_Y = BOARD_OFFSET_Y

COLORS = {
    "I": (0, 255, 255),
    "O": (255, 255, 0),
    "T": (128, 0, 128),
    "S": (0, 255, 0),
    "Z": (255, 0, 0),
    "J": (0, 0, 255),
    "L": (255, 

In [8]:
import os

print("--- BUSCANDO BLOQUEOS EN tetris_ui.py ---")
!grep -n -E 'tick\(|sleep\(|wait\(|event\.get' /content/tetris/src/tetris_ui.py

print("\n--- BUSCANDO BLOQUEOS EN confound_floor_t2.py ---")
!grep -n -E 'tick\(|sleep\(|wait\(|event\.get' /content/tetris/analysis/confound_floor/confound_floor_t2.py

--- BUSCANDO BLOQUEOS EN tetris_ui.py ---
249:            for event in pygame.event.get():
262:            self.clock.tick(60)
266:                time.sleep(1.0)

--- BUSCANDO BLOQUEOS EN confound_floor_t2.py ---


In [9]:
import os

ui_file_path = "/content/tetris/src/tetris_ui.py"

with open(ui_file_path, "r") as f:
    content = f.read()

# Reemplazamos los bloqueos para que la simulación vuele a máxima velocidad
content = content.replace("self.clock.tick(60)", "pass  # Acelerado para Colab")
content = content.replace("time.sleep(1.0)", "pass  # Acelerado para Colab")

with open(ui_file_path, "w") as f:
    f.write(content)

print("¡Parche aplicado! Los límites de FPS y las pausas han sido eliminados de tetris_ui.py.")

¡Parche aplicado! Los límites de FPS y las pausas han sido eliminados de tetris_ui.py.


In [10]:
import os

file_path = "/content/tetris/analysis/confound_floor/confound_floor_t2.py"
try:
    with open(file_path, "r") as f:
        print("--- CONTENIDO DE confound_floor_t2.py ---")
        print(f.read())
except FileNotFoundError:
    print(f"Archivo no encontrado: {file_path}")

--- CONTENIDO DE confound_floor_t2.py ---
"""
confound_floor_t2.py — Medición del piso de confound de Fase 1A en acomodación a t+2.

Escenario B del plan corregido:
- Partidas naturales con agente base bag-ciego.
- Horizonte fijado por preview=1 (decisión de colocación de t evaluada contra t+2).
- H moderado: se filtran decisiones por altura del stack para evitar survivorship.
- Clase de favorabilidad rica: conteo de tipos t+2 compatibles con el board_resultante.
- Predictor: P_tracker(fav | S_t, clase) - P_stat(fav | clase), controlando P_stat(fav|clase).
- Estimador: conditional logit sobre el conjunto de consideración (top-k por pi_fill base).
- Diagnóstico: censura on/off no debe mover el β del no-tracker a H moderado.
"""

from __future__ import annotations

import argparse
import concurrent.futures
import json
import os
import subprocess
import sys
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterator, List, Optional, Tuple



In [11]:
import os

file_path = "/content/tetris/analysis/confound_floor/confound_floor_t2.py"

with open(file_path, "r") as f:
    content = f.read()

# Reemplazamos los procesos por hilos para evitar el deadlock de Pygame al hacer fork en Colab
content = content.replace("ProcessPoolExecutor", "ThreadPoolExecutor")

with open(file_path, "w") as f:
    f.write(content)

print("¡Parche aplicado! ProcessPoolExecutor ha sido cambiado a ThreadPoolExecutor.")

¡Parche aplicado! ProcessPoolExecutor ha sido cambiado a ThreadPoolExecutor.


In [12]:
import re

file_path = "/content/tetris/analysis/confound_floor/confound_floor_t2.py"
with open(file_path, "r") as f:
    content = f.read()

print("--- DEFINICIÓN DE run_single_game_subprocess ---")
match = re.search(r"def run_single_game_subprocess.*?\n(?:\s+.*?\n)+", content)
if match:
    print(match.group(0))
else:
    print("No se encontró la función run_single_game_subprocess de forma sencilla. Mostrando una sección del código:\n")
    lines = content.split("\n")
    for i, line in enumerate(lines):
        if "def run_single_game_subprocess" in line:
            print("\n".join(lines[i:i+30]))
            break


--- DEFINICIÓN DE run_single_game_subprocess ---
No se encontró la función run_single_game_subprocess de forma sencilla. Mostrando una sección del código:



In [13]:
import os

ui_file_path = "/content/tetris/src/tetris_ui.py"
with open(ui_file_path, "r") as f:
    content = f.read()

print("--- BUCLE PRINCIPAL (run) EN tetris_ui.py ---")
lines = content.split("\n")
for i, line in enumerate(lines):
    if "def run(" in line or "while True:" in line or "while not self.game.game_over:" in line:
        print("\n".join(lines[max(0, i-2):min(len(lines), i+40)]))
        break


--- BUCLE PRINCIPAL (run) EN tetris_ui.py ---
                self._execute_action(ks.action, t_ms)

    def run(self) -> None:
        running = True
        while running:
            t_ms = self._t_ms()

            for event in pygame.event.get():
                if event.type == pygame.QUIT:
                    running = False
                elif event.type == pygame.KEYDOWN:
                    self._handle_keydown(event.key, t_ms)
                elif event.type == pygame.KEYUP:
                    self._handle_keyup(event.key, t_ms)

            self._update_das(t_ms)
            self.game.update(t_ms)
            self._check_and_log_lock(t_ms)

            self._render()
            pass  # Acelerado para Colab

            if self.game.game_over:
                # Pequeña pausa para que el jugador vea el game over.
                pass  # Acelerado para Colab
                running = False

        self._end_game()

    def _render(self) -> None:
        self.screen.fill(CO

In [14]:
import re

file_path = "/content/tetris/analysis/confound_floor/confound_floor_t2.py"
with open(file_path, "r") as f:
    content = f.read()

print("--- BUSCANDO LA FUNCIÓN WORKER EN confound_floor_t2.py ---")
lines = content.split("\n")

# Buscar dónde se usa el Executor para ver qué función se le pasa
for i, line in enumerate(lines):
    if "Executor" in line:
        print("Encontrado Executor en la línea", i+1)
        print("\n".join(lines[max(0, i-2):min(len(lines), i+10)]))
        print("-" * 40)

# Buscar la función que corre el juego (suele llamarse run_game, play, worker o simulate)
for i, line in enumerate(lines):
    if line.startswith("def ") and ("run" in line or "play" in line or "simul" in line or "worker" in line):
        print(f"\nDefinición encontrada: {line}")
        print("\n".join(lines[i:min(len(lines), i+20)]))
        print("-" * 40)


--- BUSCANDO LA FUNCIÓN WORKER EN confound_floor_t2.py ---
Encontrado Executor en la línea 898
# ---------------------------------------------------------------------------
def _simulate_one(args_tuple) -> Tuple[List[Dict], List[Dict]]:
    """Wrapper para ThreadPoolExecutor."""
    seed, params_tracker, params_no_tracker, k, H_min, H_max, no_censorship, max_pieces = args_tuple
    return simulate_natural_game(
        seed,
        params_tracker,
        params_no_tracker,
        k=k,
        H_min=H_min,
        H_max=H_max,
        no_censorship=no_censorship,
----------------------------------------
Encontrado Executor en la línea 935
                        help="Directorio de la corrida censurada de referencia para figura de desacople")
    parser.add_argument("--n_workers", type=int, default=None,
                        help="Número de workers para ThreadPoolExecutor (default: min(cpu_count,8))")
    return parser.parse_args()


def main_single_run(args: argparse.Namespace, o

In [15]:
import re

file_path = "/content/tetris/analysis/confound_floor/confound_floor_t2.py"
with open(file_path, "r") as f:
    content = f.read()

# Extraer la función simulate_natural_game
match = re.search(r"def simulate_natural_game\(.*?^def ", content, re.MULTILINE | re.DOTALL)
if match:
    func_code = match.group(0)[:-4] # Quitar el inicio del próximo def
    print("--- CÓDIGO DE simulate_natural_game ---")
    print(func_code)
else:
    # Fallback si regex falla, buscar e imprimir líneas
    lines = content.split("\n")
    in_func = False
    func_lines = []
    for line in lines:
        if line.startswith("def simulate_natural_game"):
            in_func = True
        elif in_func and line.startswith("def "):
            break

        if in_func:
            func_lines.append(line)

    print("--- CÓDIGO DE simulate_natural_game ---")
    print("\n".join(func_lines))

--- CÓDIGO DE simulate_natural_game ---
def simulate_natural_game(
    seed: int,
    params_tracker: AgentParams,
    params_no_tracker: AgentParams,
    k: int,
    H_min: int,
    H_max: int,
    no_censorship: bool = False,
    max_pieces: int = 500,
) -> Tuple[List[Dict], List[Dict]]:
    """Simula una partida natural y devuelve decisiones logueadas para tracker/no-tracker.

    El agente base genera el tablero. Luego, para cada decisión a H moderado,
    se construye el conjunto de consideración top-k y se hacen elegir a tracker
    y no-tracker. Se loguean ambas elecciones en formato largo para conditional logit.
    """
    gen = SevenBagGenerator(seed)
    rng_tracker = np.random.default_rng(seed + 1)
    rng_no_tracker = np.random.default_rng(seed + 2)

    board = empty_board()

    decisions_tracker: List[Dict] = []
    decisions_no_tracker: List[Dict] = []

    decision_id = 0

    for _ in range(max_pieces):
        piece = gen.advance()
        next_piece = gen.peek_n(ge

In [16]:
import os

file_path = "/content/tetris/analysis/confound_floor/confound_floor_t2.py"
with open(file_path, "r") as f:
    content = f.read()

# Añadir un print al inicio de cada hilo/worker
content = content.replace(
    'def _simulate_one(args_tuple) -> Tuple[List[Dict], List[Dict]]:',
    'def _simulate_one(args_tuple) -> Tuple[List[Dict], List[Dict]]:\n    print("-> [Worker] Iniciando _simulate_one", flush=True)'
)

# Añadir un print dentro del bucle de piezas para ver el progreso
content = content.replace(
    'for _ in range(max_pieces):',
    'for _ in range(max_pieces):\n        if _ % 50 == 0:\n            print(f"   [Juego {seed}] Simulando pieza {_}", flush=True)'
)

with open(file_path, "w") as f:
    f.write(content)

print("¡Prints de depuración inyectados en confound_floor_t2.py!")

¡Prints de depuración inyectados en confound_floor_t2.py!


In [17]:
import os

file_path = "/content/tetris/analysis/confound_floor/confound_floor_t2.py"
with open(file_path, "r") as f:
    content = f.read()

# Inyectar prints alrededor de pi_fill_base
content = content.replace(
    "board_after_base, info_base = pi_fill_base(board, piece)",
    "print(f'   [{_}] -> llamando pi_fill_base', flush=True)\n        board_after_base, info_base = pi_fill_base(board, piece)\n        print(f'   [{_}] <- pi_fill_base termino', flush=True)"
)

# Inyectar prints alrededor de top_k_alternatives_base
content = content.replace(
    "alts_base = top_k_alternatives_base(",
    "print(f'   [{_}] -> llamando top_k_alternatives_base', flush=True)\n            alts_base = top_k_alternatives_base("
)
content = content.replace(
    "p_tracker, chosen_idx_t, _ = pi_fill_tracker(",
    "print(f'   [{_}] <- top_k_alternatives_base termino', flush=True)\n            print(f'   [{_}] -> llamando pi_fill_tracker', flush=True)\n            p_tracker, chosen_idx_t, _ = pi_fill_tracker("
)
content = content.replace(
    "p_nt, chosen_idx_nt, _ = pi_fill_no_tracker(",
    "print(f'   [{_}] <- pi_fill_tracker termino', flush=True)\n            print(f'   [{_}] -> llamando pi_fill_no_tracker', flush=True)\n            p_nt, chosen_idx_nt, _ = pi_fill_no_tracker("
)
content = content.replace(
    "chosen_alt = allowed_alts[chosen_idx_nt]",
    "print(f'   [{_}] <- pi_fill_no_tracker termino', flush=True)\n            chosen_alt = allowed_alts[chosen_idx_nt]"
)

with open(file_path, "w") as f:
    f.write(content)

print("¡Prints detallados inyectados! Vuelve a ejecutar la celda de simulación (la que se atora) y veamos en qué función se detiene el print.")

¡Prints detallados inyectados! Vuelve a ejecutar la celda de simulación (la que se atora) y veamos en qué función se detiene el print.


In [18]:
import os
import re

# Buscar dónde se define o importa pi_fill_base
search_cmd = "grep -rn 'def pi_fill_base' /content/tetris/"
print("--- BUSCANDO def pi_fill_base ---")
os.system(search_cmd)

# También buscar si se importa en confound_floor_t2.py
print("\n--- BUSCANDO IMPORTACIÓN DE pi_fill_base EN confound_floor_t2.py ---")
os.system("grep 'pi_fill_base' /content/tetris/analysis/confound_floor/confound_floor_t2.py | grep import")

--- BUSCANDO def pi_fill_base ---

--- BUSCANDO IMPORTACIÓN DE pi_fill_base EN confound_floor_t2.py ---


256

In [19]:
file_path = "/content/tetris/analysis/confound_floor/confound_floor_t2.py"
with open(file_path, "r") as f:
    lines = f.readlines()

print("--- LÍNEAS CON 'pi_fill_base' EN confound_floor_t2.py ---")
for i, line in enumerate(lines):
    if "pi_fill_base" in line:
        print(f"Línea {i+1}: {line.strip()}")


--- LÍNEAS CON 'pi_fill_base' EN confound_floor_t2.py ---
Línea 365: def pi_fill_base(board: np.ndarray, piece: str) -> Tuple[Optional[np.ndarray], Optional[Dict]]:
Línea 490: print(f'   [{_}] -> llamando pi_fill_base', flush=True)
Línea 491: board_after_base, info_base = pi_fill_base(board, piece)
Línea 492: print(f'   [{_}] <- pi_fill_base termino', flush=True)


In [20]:
file_path = "/content/tetris/analysis/confound_floor/confound_floor_t2.py"
with open(file_path, "r") as f:
    lines = f.readlines()

print("--- CÓDIGO DE pi_fill_base ---")
for i in range(364, 400):  # Imprimir unas 35 líneas a partir de la definición
    if i < len(lines):
        print(f"{i+1}: {lines[i].rstrip()}")


--- CÓDIGO DE pi_fill_base ---
365: def pi_fill_base(board: np.ndarray, piece: str) -> Tuple[Optional[np.ndarray], Optional[Dict]]:
366:     """Coloca `piece` en `board` eligiendo la colocación con menor extended_board_value."""
367:     placements = valid_placements(board, piece)
368:     if not placements:
369:         return None, None
370:     best_board = None
371:     best_val = np.inf
372:     best_col = None
373:     for shape, col, row in placements:
374:         new_board = place_piece(board, shape, col, row)
375:         if new_board is None:
376:             continue
377:         new_board, _ = clear_lines(new_board)
378:         val = extended_board_value(new_board)
379:         if val < best_val:
380:             best_val = val
381:             best_board = new_board
382:             best_col = col
383:     if best_board is None:
384:         return None, None
385:     return best_board, {"last_landing_col": best_col}
386: 
387: 
388: def get_top_k_placements(
389:     bo

In [21]:
import os

print("--- BUSCANDO def valid_placements ---")
os.system("grep -rn 'def valid_placements' /content/tetris/")

print("\n--- BUSCANDO def place_piece ---")
os.system("grep -rn 'def place_piece' /content/tetris/")


--- BUSCANDO def valid_placements ---

--- BUSCANDO def place_piece ---


0

In [22]:
import os

print("--- BUSCANDO 'valid_placements' (amplio) ---")
os.system("grep -rn 'valid_placements' /content/tetris/src/")

print("\n--- IMPORTS EN confound_floor_t2.py ---")
os.system("grep -E '^import |^from ' /content/tetris/analysis/confound_floor/confound_floor_t2.py")


--- BUSCANDO 'valid_placements' (amplio) ---

--- IMPORTS EN confound_floor_t2.py ---


0

In [23]:
import os

file_path = "/content/tetris/analysis/confound_floor/confound_floor_t2.py"
with open(file_path, "r") as f:
    lines = f.readlines()

print("--- IMPORTS EN confound_floor_t2.py ---")
for line in lines:
    if line.startswith("import ") or line.startswith("from "):
        print(line.strip())

print("\n--- BUSCANDO valid_placements EN src/tetris_core.py ---")
core_path = "/content/tetris/src/tetris_core.py"
if os.path.exists(core_path):
    with open(core_path, "r") as f:
        core_lines = f.readlines()

    for i, line in enumerate(core_lines):
        if "valid_placements" in line:
            print(f"Línea {i+1}: {line.strip()}")
else:
    print("No se encontró tetris_core.py")


--- IMPORTS EN confound_floor_t2.py ---
from __future__ import annotations
import argparse
import concurrent.futures
import json
import os
import subprocess
import sys
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterator, List, Optional, Tuple
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.optimize as opt
import scipy.stats as st
from statsmodels.discrete.conditional_models import ConditionalLogit

--- BUSCANDO valid_placements EN src/tetris_core.py ---


In [24]:
import os

search_dir = "/content/tetris"
print(f"--- BUSCANDO 'valid_placements' EN {search_dir} ---")
for root, dirs, files in os.walk(search_dir):
    for file in files:
        if file.endswith(".py"):
            file_path = os.path.join(root, file)
            try:
                with open(file_path, "r", encoding="utf-8") as f:
                    for i, line in enumerate(f):
                        if "valid_placements" in line:
                            print(f"{file_path}:{i+1}: {line.strip()}")
            except Exception as e:
                pass


--- BUSCANDO 'valid_placements' EN /content/tetris ---
/content/tetris/analysis/confound_floor/confound_floor.py:131: def valid_placements(
/content/tetris/analysis/confound_floor/confound_floor.py:291: placements = valid_placements(board, piece, allow_well=False)
/content/tetris/analysis/confound_floor/confound_floor.py:312: all_placements = valid_placements(board, piece, allow_well=True)
/content/tetris/analysis/confound_floor/confound_floor_t2.py:125: def valid_placements(
/content/tetris/analysis/confound_floor/confound_floor_t2.py:297: for shape, col, row in valid_placements(board, piece):
/content/tetris/analysis/confound_floor/confound_floor_t2.py:367: placements = valid_placements(board, piece)
/content/tetris/analysis/confound_floor/confound_floor_t2.py:392: placements = valid_placements(board, piece)


In [25]:
import os

file_path = "/content/tetris/analysis/confound_floor/confound_floor_t2.py"
with open(file_path, "r") as f:
    lines = f.readlines()

print("--- CÓDIGO DE valid_placements --- ")
in_func = False
func_lines = []
for i in range(124, len(lines)):
    line = lines[i]
    if line.startswith("def valid_placements"):
        in_func = True
    elif in_func and line.startswith("def "):
        break

    if in_func:
        func_lines.append(f"{i+1}: {line.rstrip()}")

print("\n".join(func_lines))

print("\n--- BUSCANDO def place_piece --- ")
for i, line in enumerate(lines):
    if line.startswith("def place_piece"):
        print(f"\nEncontrado en línea {i+1}:")
        for j in range(i, min(i+40, len(lines))):
            if j > i and lines[j].startswith("def "):
                break
            print(f"{j+1}: {lines[j].rstrip()}")


--- CÓDIGO DE valid_placements --- 
125: def valid_placements(
126:     board: np.ndarray, piece: str
127: ) -> List[Tuple[np.ndarray, int, int]]:
128:     """Todas las colocaciones de aterrizaje válidas para una pieza.
129: 
130:     Cada tupla es (shape, col, row).
131:     """
132:     placements = []
133:     for shape in PIECE_ROTATIONS[piece]:
134:         h, w = shape.shape
135:         for col in range(BOARD_WIDTH - w + 1):
136:             row = landing_row(board, shape, col)
137:             if row < 0:
138:                 continue
139:             placements.append((shape, col, row))
140:     return placements
141: 
142: 

--- BUSCANDO def place_piece --- 

Encontrado en línea 104:
104: def place_piece(
105:     board: np.ndarray, shape: np.ndarray, col: int, row: int
106: ) -> Optional[np.ndarray]:
107:     """Coloca la pieza si es válida; devuelve nuevo tablero o None."""
108:     if is_collision(board, shape, col, row):
109:         return None
110:     new_board = board

In [26]:
import os

file_path = "/content/tetris/analysis/confound_floor/confound_floor_t2.py"
with open(file_path, "r") as f:
    lines = f.readlines()

functions_to_find = ["def landing_row", "def is_collision", "def clear_lines", "def extended_board_value"]

for func_name in functions_to_find:
    print(f"\n--- BUSCANDO {func_name} --- ")
    for i, line in enumerate(lines):
        if line.startswith(func_name):
            print(f"Encontrado en línea {i+1}:")
            for j in range(i, min(i+35, len(lines))):
                if j > i and lines[j].startswith("def "):
                    break
                print(f"{j+1}: {lines[j].rstrip()}")



--- BUSCANDO def landing_row --- 
Encontrado en línea 116:
116: def landing_row(board: np.ndarray, shape: np.ndarray, col: int) -> int:
117:     """Fila más baja (mayor índice) donde la pieza cabe sin colisión."""
118:     max_row = BOARD_HEIGHT - shape.shape[0]
119:     for row in range(max_row + 1):
120:         if is_collision(board, shape, col, row):
121:             return row - 1
122:     return max_row
123: 
124: 

--- BUSCANDO def is_collision --- 
Encontrado en línea 95:
95: def is_collision(board: np.ndarray, shape: np.ndarray, col: int, row: int) -> bool:
96:     """True si la pieza (esquina sup-izq en col,row) colisiona o sale del tablero."""
97:     h, w = shape.shape
98:     if col < 0 or col + w > BOARD_WIDTH or row < 0 or row + h > BOARD_HEIGHT:
99:         return True
100:     sub = board[row : row + h, col : col + w]
101:     return bool(np.any(sub & shape))
102: 
103: 

--- BUSCANDO def clear_lines --- 
Encontrado en línea 143:
143: def clear_lines(board: np.ndarray

In [27]:
import os

file_path = "/content/tetris/analysis/confound_floor/confound_floor_t2.py"
with open(file_path, "r") as f:
    lines = f.readlines()

functions_to_find = ["def p_t2_distribution", "def current_bag_state", "def column_heights", "def top_k_alternatives_base"]

for func_name in functions_to_find:
    print(f"\n--- BUSCANDO {func_name} --- ")
    found = False
    for i, line in enumerate(lines):
        if line.startswith(func_name):
            found = True
            print(f"Encontrado en línea {i+1}:")
            for j in range(i, min(i+40, len(lines))):
                if j > i and lines[j].startswith("def "):
                    break
                print(f"{j+1}: {lines[j].rstrip()}")
    if not found:
        print("No se encontró en este archivo.")



--- BUSCANDO def p_t2_distribution --- 
Encontrado en línea 322:
322: def p_t2_distribution(gen: SevenBagGenerator) -> Dict[str, float]:
323:     """Distribución de la pieza en t+2 dado el estado actual del generador.
324: 
325:     El generador está posicionado en t+1 (la próxima pieza a consumir).
326:     t+2 es la siguiente. Si t+1 es la última del bag, t+2 es del siguiente bag
327:     y es uniforme sobre las 7 piezas.
328:     """
329:     dist = {p: 0.0 for p in BAG}
330:     if gen.idx % 7 == 6:
331:         # t+1 es el último del bag; t+2 proviene del siguiente bag (desconocido)
332:         for p in BAG:
333:             dist[p] = 1.0 / 7.0
334:     else:
335:         # t+2 es uniforme entre las piezas no vistas del bag actual, excluyendo t+1
336:         S_t = current_bag_state(gen)
337:         next_piece = gen.sequence[gen.idx]
338:         candidates = S_t - {next_piece}
339:         if len(candidates) == 0:
340:             # fallback improbable: uniforme
341:          

In [28]:
import os

file_path = "/content/tetris/analysis/confound_floor/confound_floor_t2.py"
with open(file_path, "r") as f:
    lines = f.readlines()

print("--- BUSCANDO SevenBagGenerator en confound_floor_t2.py ---")
found = False
for i, line in enumerate(lines):
    if "class SevenBagGenerator" in line or "def SevenBagGenerator" in line:
        found = True
        print(f"Encontrado en línea {i+1}:")
        for j in range(i, min(i+40, len(lines))):
            if j > i and (lines[j].startswith("class ") or lines[j].startswith("def ")):
                break
            print(f"{j+1}: {lines[j].rstrip()}")
if not found:
    print("No está definido en este archivo. Buscando en src/tetris_core.py...")
    core_path = "/content/tetris/src/tetris_core.py"
    if os.path.exists(core_path):
        with open(core_path, "r") as f:
            core_lines = f.readlines()
        for i, line in enumerate(core_lines):
            if "class SevenBagGenerator" in line:
                print(f"Encontrado en {core_path} línea {i+1}:")
                for j in range(i, min(i+40, len(core_lines))):
                    if j > i and (core_lines[j].startswith("class ") or core_lines[j].startswith("def ")):
                        break
                    print(f"{j+1}: {core_lines[j].rstrip()}")


--- BUSCANDO SevenBagGenerator en confound_floor_t2.py ---
Encontrado en línea 244:
244: class SevenBagGenerator:
245:     def __init__(self, seed: int, length: int = 5000):
246:         self.rng = np.random.default_rng(seed)
247:         self.sequence: List[str] = []
248:         self.idx = 0
249:         while len(self.sequence) < length:
250:             bag = list(BAG)
251:             self.rng.shuffle(bag)
252:             self.sequence.extend(bag)
253: 
254:     def advance(self) -> str:
255:         piece = self.sequence[self.idx]
256:         self.idx += 1
257:         return piece
258: 
259:     def peek_n(self, start: int, n: int) -> List[str]:
260:         """Mira n piezas desde start sin consumirlas."""
261:         return self.sequence[start : start + n]
262: 
263: 


In [29]:
import os
import re

file_path = "/content/tetris/analysis/confound_floor/confound_floor_t2.py"
with open(file_path, "r") as f:
    content = f.read()

# Añadir prints a get_top_k_placements y las funciones de los agentes
content = re.sub(
    r"(alts_base\s*=\s*get_top_k_placements\()",
    r"print(f'   [{_}] -> llamando get_top_k_placements', flush=True)\n            \1",
    content
)

content = re.sub(
    r"(p_tracker,\s*chosen_idx_t,\s*_\s*=\s*pi_fill_tracker\()",
    r"print(f'   [{_}] <- get_top_k_placements termino', flush=True)\n            print(f'   [{_}] -> llamando pi_fill_tracker', flush=True)\n            \1",
    content
)

content = re.sub(
    r"(p_nt,\s*chosen_idx_nt,\s*_\s*=\s*pi_fill_no_tracker\()",
    r"print(f'   [{_}] <- pi_fill_tracker termino', flush=True)\n            print(f'   [{_}] -> llamando pi_fill_no_tracker', flush=True)\n            \1",
    content
)

content = re.sub(
    r"(chosen_alt\s*=\s*allowed_alts\[chosen_idx_nt\])",
    r"print(f'   [{_}] <- pi_fill_no_tracker termino', flush=True)\n            \1",
    content
)

with open(file_path, "w") as f:
    f.write(content)

print("¡Prints de depuración corregidos! Corre de nuevo la celda de la simulación.")


¡Prints de depuración corregidos! Corre de nuevo la celda de la simulación.


In [30]:
import os

file_path = "/content/tetris/analysis/confound_floor/confound_floor_t2.py"
with open(file_path, "r") as f:
    lines = f.read().split("\n")

print("--- LÍNEAS DESPUÉS DE pi_fill_base ---")
for i, line in enumerate(lines):
    if "Conjunto de consideración:" in line:
        for j in range(max(0, i-2), min(len(lines), i+30)):
            print(f"{j+1}: {lines[j]}")
        break


--- LÍNEAS DESPUÉS DE pi_fill_base ---
508:             t2_dist = p_t2_distribution(gen)
509: 
510:             # Conjunto de consideración: top-k por pi_fill base
511:             top_placements = get_top_k_placements(board, piece, k)
512:             if len(top_placements) >= 2:
513:                 # Tracker: usa S_t con probabilidad tracker_prob; con probabilidad
514:                 # complementaria usa la creencia estacionaria (simula tracking imperfecto).
515:                 use_tracking = rng_tracker.random() < params_tracker.tracker_prob
516: 
517:                 def tracker_t2_term(res_board: np.ndarray) -> float:
518:                     _, compatible = compatibility_class(res_board)
519:                     return p_favorable_given_class(compatible, t2_dist)
520: 
521:                 def no_tracker_t2_term(res_board: np.ndarray) -> float:
522:                     count, _ = compatibility_class(res_board)
523:                     return p_stat_favorable_class(count)
524: 

In [31]:
import os
import re

file_path = "/content/tetris/analysis/confound_floor/confound_floor_t2.py"
with open(file_path, "r") as f:
    content = f.read()

# Buscamos el bloque del ThreadPoolExecutor y lo reemplazamos por un loop síncrono
old_block = r"""        else:
            with concurrent.futures.ThreadPoolExecutor(max_workers=n_workers) as executor:
                for dec_t, dec_nt in executor.map(_simulate_one, arg_tuples, chunksize=max(1, args.n_games // n_workers)):
                    all_decisions.extend(dec_t)
                    all_decisions.extend(dec_nt)"""

new_block = """        else:
            for args_t in arg_tuples:
                dec_t, dec_nt = _simulate_one(args_t)
                all_decisions.extend(dec_t)
                all_decisions.extend(dec_nt)"""

if old_block in content:
    content = content.replace(old_block, new_block)
    with open(file_path, "w") as f:
        f.write(content)
    print("¡Parche aplicado! Se eliminó el ThreadPoolExecutor para forzar el error síncrono.")
else:
    print("No se encontró el bloque exacto. Usando regex...")
    content = re.sub(
        r"with concurrent\.futures\.ThreadPoolExecutor.*?all_decisions\.extend\(dec_nt\)",
        "for args_t in arg_tuples:\n                dec_t, dec_nt = _simulate_one(args_t)\n                all_decisions.extend(dec_t)\n                all_decisions.extend(dec_nt)",
        content,
        flags=re.DOTALL
    )
    with open(file_path, "w") as f:
        f.write(content)
    print("Regex aplicado.")

print("\nVuelve a correr la simulación (se atorará, pero esta vez debería lanzar un ERROR con el Traceback real).")

¡Parche aplicado! Se eliminó el ThreadPoolExecutor para forzar el error síncrono.

Vuelve a correr la simulación (se atorará, pero esta vez debería lanzar un ERROR con el Traceback real).


In [32]:
import os

file_path = "/content/tetris/analysis/confound_floor/confound_floor_t2.py"
with open(file_path, "r") as f:
    content = f.read()

# Inyectar prints correctos
content = content.replace(
    "top_placements = get_top_k_placements(board, piece, k)",
    "print(f'   [{_}] -> llamando get_top_k_placements', flush=True)\n            top_placements = get_top_k_placements(board, piece, k)\n            print(f'   [{_}] <- get_top_k_placements termino', flush=True)"
)

content = content.replace(
    "chosen_idx_t, _ = choose_among_placements(",
    "print(f'   [{_}] -> llamando choose_among_placements (tracker)', flush=True)\n                chosen_idx_t, _ = choose_among_placements("
)

content = content.replace(
    "chosen_idx_nt, _ = choose_among_placements(",
    "print(f'   [{_}] <- choose_among_placements (tracker) termino', flush=True)\n                print(f'   [{_}] -> llamando choose_among_placements (no_tracker)', flush=True)\n                chosen_idx_nt, _ = choose_among_placements("
)

content = content.replace(
    "for idx, (shape, col, row, base_val) in enumerate(top_placements):",
    "print(f'   [{_}] <- choose_among_placements (no_tracker) termino', flush=True)\n                for idx, (shape, col, row, base_val) in enumerate(top_placements):"
)

with open(file_path, "w") as f:
    f.write(content)

print("¡Prints corregidos e inyectados! Vuelve a correr la simulación.")

¡Prints corregidos e inyectados! Vuelve a correr la simulación.


In [33]:
import os

file_path = "/content/tetris/analysis/confound_floor/confound_floor_t2.py"
with open(file_path, "r") as f:
    lines = f.readlines()

print("--- CÓDIGO DE LAS FUNCIONES LENTAS ---\n")

# Imprimir compatibility_class y funciones asociadas al t+2
for i in range(270, 330):
    if i < len(lines):
        print(f"{i+1}: {lines[i].rstrip()}")

print("\n" + "="*50 + "\n")

# Imprimir choose_among_placements
for i in range(400, 460):
    if i < len(lines):
        print(f"{i+1}: {lines[i].rstrip()}")

--- CÓDIGO DE LAS FUNCIONES LENTAS ---

271: # ---------------------------------------------------------------------------
272: # Favorabilidad local del board_resultante para t+2 (clase rica)
273: # ---------------------------------------------------------------------------
274: def count_holes(board: np.ndarray) -> int:
275:     """Número de celdas vacías con al menos una celda ocupada arriba en la columna.
276: 
277:     Vectorizado con numpy.
278:     """
279:     # Para columnas vacías, no hay huecos.
280:     has_occupied = board.any(axis=0)
281:     first_occ = np.argmax(board, axis=0)
282:     rows = np.arange(BOARD_HEIGHT)[:, None]
283:     below_top = rows >= first_occ[None, :]
284:     empty = board == 0
285:     holes = below_top & empty & has_occupied[None, :]
286:     return int(np.sum(holes))
287: 
288: 
289: def piece_fits_clean(board: np.ndarray, piece: str) -> bool:
290:     """True si `piece` puede colocarse en `board` sin aumentar el número de huecos.
291: 
292:    

In [34]:
import os
import re
import subprocess

conf_dir = "/content/tetris/analysis/confound_floor"
file_path = os.path.join(conf_dir, "confound_floor_t2.py")

# 1. Restaurar el archivo original para borrar los prints lentos
subprocess.run(["git", "checkout", "confound_floor_t2.py"], cwd=conf_dir)

with open(file_path, "r") as f:
    content = f.read()

# 2. Parche para evitar el bloqueo de procesos en Colab
content = content.replace("ProcessPoolExecutor", "ThreadPoolExecutor")

# 3. Optimización matemática extrema para count_holes
old_count = r"def count_holes\(board: np\.ndarray\) -> int:.*?return int\(np\.sum\(holes\)\)"
new_count = """def count_holes(board: np.ndarray) -> int:
    # Optimizado matemáticamente con suma acumulada vectorial
    return int(np.sum((np.cumsum(board, axis=0) > 0) & (board == 0)))"""
content = re.sub(old_count, new_count, content, flags=re.DOTALL)

# 4. Añadir memoria caché a compatibility_class para no evaluar el mismo tablero miles de veces
old_compat = r"def compatibility_class\(board: np\.ndarray\) -> Tuple\[int, List\[str\]\]:.*?return len\(compatible\), compatible"
new_compat = """_compat_cache = {}
def compatibility_class(board: np.ndarray) -> Tuple[int, List[str]]:
    b_bytes = board.tobytes()
    if b_bytes in _compat_cache:
        return _compat_cache[b_bytes]
    compatible = [p for p in BAG if piece_fits_clean(board, p)]
    res = (len(compatible), compatible)
    if len(_compat_cache) > 50000:
        _compat_cache.clear()  # Evitar que explote la RAM
    _compat_cache[b_bytes] = res
    return res"""
content = re.sub(old_compat, new_compat, content, flags=re.DOTALL)

with open(file_path, "w") as f:
    f.write(content)

print("¡Super parche de rendimiento aplicado con éxito!\n")
print("Ve a la celda de la simulación y córrela de nuevo. Ahora sí va a volar 🚀")

¡Super parche de rendimiento aplicado con éxito!

Ve a la celda de la simulación y córrela de nuevo. Ahora sí va a volar 🚀


In [35]:
import os

file_path = "/content/tetris/analysis/confound_floor/confound_floor_t2.py"
with open(file_path, "r") as f:
    content = f.read()

# Agregar un punto por cada worker/partida iniciada para ver progreso sin spam
old_str = "def _simulate_one(args_tuple) -> Tuple[List[Dict], List[Dict]]:"
new_str = "def _simulate_one(args_tuple) -> Tuple[List[Dict], List[Dict]]:\n    print('.', end='', flush=True)"

if "print('.', end='', flush=True)" not in content:
    content = content.replace(old_str, new_str)
    with open(file_path, "w") as f:
        f.write(content)
    print("¡Indicador de progreso (puntitos) añadido! Corre tu simulación de nuevo.")
else:
    print("El indicador ya estaba añadido.")

¡Indicador de progreso (puntitos) añadido! Corre tu simulación de nuevo.


In [36]:
import subprocess
import os
import sys
import time

conf_dir = "/content/tetris/analysis/confound_floor"
out_dir = os.path.join(conf_dir, "out_test_rapido")

cmd = [
    sys.executable, "-u", "confound_floor_t2.py",
    "--n_games", "2",  # Solo 2 partidas para probar que no se cuelga
    "--max_pieces", "50", # Pocas piezas para que termine instantáneo
    "--tau", "10.0",
    "--H_min", "4",
    "--H_max", "15",
    "--tracker_prob", "0.50",
    "--no_censorship",
    "--out", out_dir,
    "--n_workers", "1"
]

print("Iniciando prueba rápida (2 partidas, 50 piezas)...")
t0 = time.time()

# Leemos carácter por carácter para que los puntitos sí se vean en tiempo real
process = subprocess.Popen(cmd, cwd=conf_dir, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

while True:
    char = process.stdout.read(1)
    if not char and process.poll() is not None:
        break
    if char:
        sys.stdout.write(char)
        sys.stdout.flush()

elapsed = time.time() - t0
print(f"\n\n¡Prueba terminada exitosamente en {elapsed:.2f} segundos!")
print("Esto demuestra que el motor matemático ya vuela y no se cuelga 🚀")

Iniciando prueba rápida (2 partidas, 50 piezas)...
[k=5] Simulando 2 partidas...
    Usando 1 worker(s) paralelo(s)
..    Decisiones logueadas: tracker=360, no_tracker=360, decisiones únicas=72
    Varianza intra-decisión p_stat_clase: mean_std=0.0645, frac_zero_std=0.1667
    no_tracker bruto:  -1.1400 (CI -3.9505, 1.6705), p=0.4266
    no_tracker oracle: -1.1274 (CI -3.9812, 1.7264), p=0.4387
    tracker bruto:     9.0598 (CI 2.0046, 16.1149), p=0.01184
    tracker oracle:    8.9898 (CI 2.9326, 15.0471), p=0.003627
    rama: 1_piso_limpio

--- Resultado ---
no_tracker bruto:  -1.1400 (CI -3.9505, 1.6705), p=0.4266
no_tracker oracle: -1.1274 (CI -3.9812, 1.7264), p=0.4387
tracker bruto:     9.0598 (CI 2.0046, 16.1149), p=0.01184
tracker oracle:    8.9898 (CI 2.9326, 15.0471), p=0.003627
Rama: 1_piso_limpio
Salidas en: /content/tetris/analysis/confound_floor/out_test_rapido


¡Prueba terminada exitosamente en 11.48 segundos!
Esto demuestra que el motor matemático ya vuela y no se cuelg

In [37]:
import subprocess
import os
import time
import sys
import multiprocessing

cores = multiprocessing.cpu_count()
print(f"-> Utilizando {cores} núcleos de CPU (Colab Pro) para la simulación principal.\n")

conf_dir = os.path.join(PROJECT_DIR, "analysis", "confound_floor")
python = sys.executable

def run_simulation(tp, n_games=N_GAMES):
    tp_str = f"{int(tp*100):03d}"
    out_dir = f"out_t2_H15_n{n_games}_tp{tp_str}_nocensor"
    out_path = os.path.join(conf_dir, out_dir)
    cmd = [
        python, "-u", "confound_floor_t2.py",
        "--n_games", str(n_games),
        "--max_pieces", str(MAX_PIECES),
        "--tau", str(TAU),
        "--H_min", str(H_MIN),
        "--H_max", str(H_MAX),
        "--tracker_prob", str(tp),
        "--no_censorship",
        "--out", out_path,
        "--n_workers", str(cores)  # Usa todos los cores de Colab Pro
    ]
    print(f"\n[tracker_prob={tp}] Iniciando...")
    t0 = time.time()

    # Configurar entorno headless para evitar que pygame se congele en Colab
    env = os.environ.copy()
    env["SDL_VIDEODRIVER"] = "dummy"
    env["SDL_AUDIODRIVER"] = "dummy"

    try:
        # Usar Popen para transmitir la salida en tiempo real, pasando el nuevo env
        process = subprocess.Popen(cmd, cwd=conf_dir, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
        for line in process.stdout:
            sys.stdout.write(line)
            sys.stdout.flush()

        process.wait()

        if process.returncode != 0:
            print(f"ERROR (returncode={process.returncode})")
        else:
            elapsed = time.time() - t0
            print(f"[tracker_prob={tp}] OK en {elapsed/60:.1f} min")
        return process.returncode == 0
    except subprocess.TimeoutExpired:
        print(f"ERROR: [tracker_prob={tp}] La simulación ha excedido el tiempo límite.")
        process.kill()
        return False

# Correr simulaciones
for tp in TRACKER_PROBS:
    run_simulation(tp)

-> Utilizando 8 núcleos de CPU (Colab Pro) para la simulación principal.


[tracker_prob=0.1] Iniciando...
[k=5] Simulando 50 partidas...
    Usando 8 worker(s) paralelo(s)
..................................................    Decisiones logueadas: tracker=75240, no_tracker=75240, decisiones únicas=15048
    Varianza intra-decisión p_stat_clase: mean_std=0.0612, frac_zero_std=0.2279
    no_tracker bruto:  -0.1662 (CI -0.3914, 0.0591), p=0.1482
    no_tracker oracle: -0.0871 (CI -0.3267, 0.1525), p=0.4761
    tracker bruto:     0.4792 (CI 0.2441, 0.7143), p=6.466e-05
    tracker oracle:    0.6460 (CI 0.3968, 0.8952), p=3.752e-07
    rama: 1_piso_limpio

--- Resultado ---
no_tracker bruto:  -0.1662 (CI -0.3914, 0.0591), p=0.1482
no_tracker oracle: -0.0871 (CI -0.3267, 0.1525), p=0.4761
tracker bruto:     0.4792 (CI 0.2441, 0.7143), p=6.466e-05
tracker oracle:    0.6460 (CI 0.3968, 0.8952), p=3.752e-07
Rama: 1_piso_limpio
Salidas en: /content/tetris/analysis/confound_floor/out_t2_H15_n50_

In [38]:
import subprocess
import os
import sys
import time
import multiprocessing

cores = multiprocessing.cpu_count()
print(f"-> Google Colab nos ha asignado {cores} núcleos de CPU.\n")

conf_dir = "/content/tetris/analysis/confound_floor"
out_dir = os.path.join(conf_dir, "out_test_rapido_2workers")

cmd = [
    sys.executable, "-u", "confound_floor_t2.py",
    "--n_games", "2",
    "--max_pieces", "50",
    "--tau", "10.0",
    "--H_min", "4",
    "--H_max", "15",
    "--tracker_prob", "0.50",
    "--no_censorship",
    "--out", out_dir,
    "--n_workers", "2"  # Probando con 2 workers
]

print("Iniciando prueba rápida (2 partidas) con 2 WORKERS...")
t0 = time.time()

process = subprocess.Popen(cmd, cwd=conf_dir, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

while True:
    char = process.stdout.read(1)
    if not char and process.poll() is not None:
        break
    if char:
        sys.stdout.write(char)
        sys.stdout.flush()

elapsed = time.time() - t0
print(f"\n\n¡Prueba con 2 workers terminada exitosamente en {elapsed:.2f} segundos!")

-> Google Colab nos ha asignado 8 núcleos de CPU.

Iniciando prueba rápida (2 partidas) con 2 WORKERS...
[k=5] Simulando 2 partidas...
    Usando 2 worker(s) paralelo(s)
..    Decisiones logueadas: tracker=360, no_tracker=360, decisiones únicas=72
    Varianza intra-decisión p_stat_clase: mean_std=0.0645, frac_zero_std=0.1667
    no_tracker bruto:  -1.1400 (CI -3.9505, 1.6705), p=0.4266
    no_tracker oracle: -1.1274 (CI -3.9812, 1.7264), p=0.4387
    tracker bruto:     9.0598 (CI 2.0046, 16.1149), p=0.01184
    tracker oracle:    8.9898 (CI 2.9326, 15.0471), p=0.003627
    rama: 1_piso_limpio

--- Resultado ---
no_tracker bruto:  -1.1400 (CI -3.9505, 1.6705), p=0.4266
no_tracker oracle: -1.1274 (CI -3.9812, 1.7264), p=0.4387
tracker bruto:     9.0598 (CI 2.0046, 16.1149), p=0.01184
tracker oracle:    8.9898 (CI 2.9326, 15.0471), p=0.003627
Rama: 1_piso_limpio
Salidas en: /content/tetris/analysis/confound_floor/out_test_rapido_2workers


¡Prueba con 2 workers terminada exitosamente en 

## 5. Análisis de potencia (lineal)

In [39]:
# Usar la corrida tracker_prob=0.5 como base
base_dir = os.path.join(conf_dir, f"out_t2_H15_n{N_GAMES}_tp050_nocensor")
power_out = os.path.join(conf_dir, "out_power_t2")

cmd = [
    python, "power_curve_t2.py",
    "--results_dir", base_dir,
    "--out", power_out
]
result = subprocess.run(cmd, cwd=conf_dir, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)

Cargando datos de /content/tetris/analysis/confound_floor/out_t2_H15_n50_tp050_nocensor/decisions_log_k5.parquet...
Piezas por partida usadas para conversion: 500.0 (no_censorship=True, max_pieces=500)
Calculando regresiones por bin de H...
  H=4-6: n_dec=5831, tracker_bruto beta=3.33 CI(2.80,3.86), piso_oracle beta=0.02 CI(-0.38,0.43)
  H=7-8: n_dec=2485, tracker_bruto beta=3.17 CI(2.42,3.92), piso_oracle beta=0.11 CI(-0.48,0.71)
  H=9-10: n_dec=2102, tracker_bruto beta=2.70 CI(1.93,3.46), piso_oracle beta=-0.36 CI(-1.02,0.29)
  H=11-12: n_dec=1981, tracker_bruto beta=3.16 CI(2.40,3.92), piso_oracle beta=-0.42 CI(-1.01,0.17)
  H=13-15: n_dec=2649, tracker_bruto beta=3.16 CI(2.46,3.87), piso_oracle beta=-0.05 CI(-0.61,0.52)

Construyendo curvas de potencia...
Generando figuras...

PREGUNTA QUE LA CURVA RESPONDE:
Cual es el tracker_prob minimo detectable con el N que una campaña
humana realista entrega, y esta por debajo del tracking humano plausible?

RESPUESTA:
Con una campaña de 10-1

## 6. Calibración rápida de la forma de β_señal(p)

In [40]:
import pandas as pd
import numpy as np

# Crear muestra del 20% del log p=0.5 para acelerar la calibración
log_path = os.path.join(base_dir, "decisions_log_k5.parquet")
df = pd.read_parquet(log_path)

bins = [(4,6,'4-6'),(7,8,'7-8'),(9,10,'9-10'),(11,12,'11-12'),(13,15,'13-15')]
sampled_ids = []
for hmin, hmax, label in bins:
    ids = df[(df['H']>=hmin)&(df['H']<=hmax)]['decision_id'].unique()
    n_sample = max(100, int(len(ids)*0.2))
    sampled_ids.extend(np.random.choice(ids, size=min(n_sample, len(ids)), replace=False))

df_sample = df[df['decision_id'].isin(sampled_ids)].copy()
id_map = {old: new for new, old in enumerate(sorted(df_sample['decision_id'].unique()))}
df_sample['decision_id'] = df_sample['decision_id'].map(id_map)
sample_path = os.path.join(base_dir, "decisions_log_k5_sample20.parquet")
df_sample.to_parquet(sample_path)
print(f"Muestra guardada: {df_sample['decision_id'].nunique()} decisiones")

# Calibración rápida
cal_out = os.path.join(conf_dir, "out_fast_calibrate_v2")
bin_results = os.path.join(power_out, "bin_results.json")
cmd = [
    python, "fast_calibrate_signal_v2.py",
    "--log", sample_path,
    "--real_results", bin_results,
    "--out", cal_out,
    "--n_replicates", "5"
]
result = subprocess.run(cmd, cwd=conf_dir, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)

Muestra guardada: 3008 decisiones
Calibracion guardada en /content/tetris/analysis/confound_floor/out_fast_calibrate_v2



In [41]:
# Recalcular p_min con curva calibrada
cal_json = os.path.join(cal_out, "fast_calibration_v2.json")
cmd = [
    python, "recalc_power_calibrated.py",
    "--bin_results", bin_results,
    "--calibration", cal_json
]
result = subprocess.run(cmd, cwd=conf_dir, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)

p_min por bin y escenario: LINEAL vs CALIBRADO
Bin      Escenario            p_min lineal    p_min calibrado forma b   
--------------------------------------------------------------------------------
4-6      campaña              0.33-0.78       0.33-0.80       0.96      
7-8      campaña              0.52->1         0.52->1         0.97      
9-10     campaña              0.56->1         0.56->1         0.98      
11-12    campaña              0.43->1         0.45-0.88       1.38      
13-15    campaña              0.45->1         0.45-0.97       1.13      

Nota: p_min calibrado usa beta(p) = beta(0.5) * (p/0.5)^b_shape.
b<1 indica curva concava (mas señal a p bajo de lo que predice lineal).



## 7. Empaquetar y descargar resultados

In [42]:
import shutil
import os

zip_path = os.path.join(PROJECT_DIR, "results_colab")
shutil.make_archive(
    zip_path, "zip",
    root_dir=conf_dir,
    base_dir="."
)
print(f"Zip creado: {zip_path}.zip")
print(f"Tamaño: {os.path.getsize(zip_path+'.zip')/1e6:.1f} MB")

Zip creado: /content/tetris/results_colab.zip
Tamaño: 64.1 MB


In [43]:
from google.colab import files
files.download(f"{zip_path}.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Nota de interpretación preliminar

Si los resultados replican lo observado en local, el `p_min` detectable con una campaña humana realista estará alrededor de 0.66–0.70 en el mejor bin (H=4–6) y `>1` en los demás. Eso apunta a que el cuello de botella es el número de decisiones informativas por sesión, no la extrapolación lineal de `β_señal(p)`.

Antes de saltar a un probe exógeno, la palanca sobre el denominador es densificar decisiones: concentrar en H=4–6, sesiones más largas, o un régimen que genere más estados con ≥2 alternativas viables y bolsa no degenerada.